# 评测方法论：快了以后，模型还是“够好”的吗？

> 推理优化不是只看 tokens/s。量化、不同采样参数、不同引擎、不同 Prompt 模板，都可能改变最终结果。
>
> 这一章只回答一个问题：**怎么做一场可信、可复现、能解释差异的评测？**
>
> 读完后，你应该能看懂模型报告里的 benchmark 表，也能自己比较 BF16 / INT8 / INT4、不同引擎或不同 serving 配置。


## 1. 一次评测其实是一条 Pipeline

```text
Dataset / Benchmark
        ↓
Prompt / Chat Template
        ↓
Generation config
        ↓
Model / Engine
        ↓
Parser
        ↓
Metric or Judge
        ↓
Aggregation + confidence interval
```

任何一环不一致，“模型 A 比模型 B 高 1 分”都可能没有意义。


## 2. 先分清你在评什么

- **Model quality**：知识、数学、代码、指令遵循、对话、安全……
- **Inference quality regression**：量化 / Kernel / 引擎改变后有没有掉点
- **Serving performance**：TTFT、TPOT、throughput、P50/P95 latency、显存
- **System cost**：每百万 Token 成本、GPU 数量、功耗等

厂商报告常把 quality 和 performance 放在同一张图上，因为工程优化本质上是 trade-off。


In [ ]:
configs = {
    "BF16": {"memory_gb":14.0, "quality":72.4, "throughput":1.0},
    "INT8": {"memory_gb":7.2,  "quality":72.2, "throughput":1.35},
    "INT4": {"memory_gb":3.8,  "quality":71.3, "throughput":1.75},
}
for name,v in configs.items():
    print(name, v)
print("真正的问题不是 INT4 快不快，而是：省下的显存/吞吐，值不值得质量下降。")


## 3. Benchmark 和 Metric 不是一回事

Benchmark 定义题目，Metric 定义怎么得分。

例如：
- MMLU / GPQA：accuracy
- GSM8K / MATH：exact match 或答案解析
- HumanEval / LiveCodeBench：pass@k / execution
- SWE-bench：任务是否真正通过测试
- 开放式对话：pairwise preference / LLM-as-Judge / human eval

同一个 Benchmark 换 Prompt、few-shot、parser，分数都可能变化。


## 4. LLM-as-Judge：方便，但不是“真理机”

常见偏差：
- position bias
- verbosity / length bias
- style bias
- self-preference
- rubric 不清

所以至少要固定 rubric、随机交换 A/B 顺序，并保留原始 judge 输出以便审计。


## 5. 为什么要置信区间？

20 道题 14/20 和 15/20 的差距，远不足以证明后者稳定更强。

下面用 bootstrap 直观看随机波动。


In [ ]:
import random, statistics

random.seed(42)
scores = [1]*15 + [0]*5

boots=[]
for _ in range(5000):
    sample=[random.choice(scores) for _ in scores]
    boots.append(sum(sample)/len(sample))

boots.sort()
print("accuracy:", sum(scores)/len(scores))
print("95% bootstrap interval:", round(boots[125],3), round(boots[-126],3))


## 6. 工具怎么选？把工具放回 Pipeline，而不是背排行榜

- **lm-evaluation-harness / OpenCompass 等**：标准 benchmark pipeline
- **AlpacaEval / MT-Bench 类**：对话 preference / judge
- **SWE-bench harness**：代码 Agent / repo-level execution
- **Promptfoo / DeepEval 等**：应用层回归 / CI
- **vLLM benchmark / serving benchmark**：系统吞吐与延迟

工具会变，Pipeline 不会。


## 7. 一个生产可用的最小对比

假设你要决定是否从 BF16 换到 AWQ INT4：

```text
固定：
- model revision
- tokenizer / chat template
- dataset
- generation config
- max context / max output
- hardware
- concurrency

同时记录：
Quality: benchmark score
Memory : peak GPU memory
TTFT   : P50 / P95
TPOT   : P50 / P95
TPS    : throughput
```

只有这样才能回答：“INT4 到底值不值得上。”


下一章不再讲指标和算法，而是完成最后一公里：

> **把 checkpoint 真正交给 vLLM / SGLang，暴露 OpenAI-compatible API，并做一次最小 serving benchmark。**
